# Lab 02: Webhook Integration — Testing & Troubleshooting

**Duration**: ~15 minutes  
**Prerequisites**: Completed `05_webhook_server.ipynb` (Flask server + ngrok tunnel must still be running)

## Learning Objectives

By the end of this notebook, you will:
- Trigger every major event type programmatically and verify delivery
- Understand the Stripe CLI `stripe trigger` workflow and its Python API equivalent
- Use the Dashboard to inspect, filter, and resend events
- Know how to troubleshoot common webhook failures

---

## Note on the Stripe CLI

The original workshop uses three terminals and the Stripe CLI:

```
Terminal 1: flask --app src/server run --port=4242
Terminal 2: stripe listen --forward-to localhost:4242/webhook
Terminal 3: stripe trigger payment_intent.succeeded
```

In Colab we use the equivalent Python API approach — creating real Stripe objects that fire the same events. The table below maps CLI triggers to their API equivalents:

| `stripe trigger` event | Python API equivalent |
|------------------------|-----------------------|
| `payment_intent.succeeded` | `stripe.PaymentIntent.create(..., confirm=True)` |
| `payment_intent.payment_failed` | Create PI with a declining card |
| `charge.refunded` | `stripe.Refund.create(charge=...)` |
| `customer.created` | `stripe.Customer.create(...)` |
| `customer.subscription.created` | `stripe.Subscription.create(...)` |
| `invoice.paid` | Triggered automatically when a subscription starts |

> **If you prefer the CLI:** Install it locally (`brew install stripe/stripe-cli/stripe`), run `stripe login`, then `stripe listen --forward-to localhost:4242/webhook` and `stripe trigger <event_type>`.

## Setup

If your Flask server from `05_webhook_server.ipynb` is still running in this same Colab session, `received_events` is already populated. If you are starting fresh, run the setup below.

In [1]:
!pip install stripe flask pyngrok --quiet

import json
import time
import threading
import stripe
from flask import Flask, jsonify, request

try:
    from google.colab import userdata
    stripe.api_key = userdata.get('STRIPE_SECRET_KEY')
    print("Loaded API key from Colab Secrets.")
except Exception:
    import getpass
    stripe.api_key = getpass.getpass("Paste your Stripe test secret key (sk_test_...): ")

try:
    account = stripe.Account.retrieve()
    print(f"Connected: {account.id}")
except stripe.error.AuthenticationError:
    print("ERROR: Invalid API key.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 21.0 MB/s eta 0:00:00
Loaded API key from Colab Secrets.
Connected: acct_1RnL4mBMxfUzotEq


In [2]:
# Skip this cell if the server from 05_webhook_server.ipynb is still running.
# Run it if you are starting this notebook fresh.

app = Flask(__name__)
received_events = []
WEBHOOK_SECRET = None

@app.route('/webhook', methods=['POST'])
def webhook():
    payload    = request.data
    sig_header = request.headers.get('Stripe-Signature')

    if WEBHOOK_SECRET:
        try:
            event = stripe.Webhook.construct_event(payload, sig_header, WEBHOOK_SECRET)
        except stripe.error.SignatureVerificationError as e:
            print(f'Signature verification failed: {e}')
            return jsonify(success=False), 400
    else:
        try:
            event = json.loads(payload)
        except json.JSONDecodeError:
            return jsonify(success=False), 400

    received_events.append(event)
    event_type = event['type'] if isinstance(event, dict) else event.type
    print(f'[webhook] {event_type}')
    return jsonify(success=True)

flask_thread = threading.Thread(
    target=lambda: app.run(port=4242, debug=False, use_reloader=False)
)
flask_thread.daemon = True
flask_thread.start()

from pyngrok import ngrok
public_url = ngrok.connect(4242).public_url
WEBHOOK_URL = f"{public_url}/webhook"

endpoint = stripe.WebhookEndpoint.create(
    url=WEBHOOK_URL,
    enabled_events=[
        "payment_intent.succeeded",
        "payment_intent.payment_failed",
        "charge.refunded",
        "customer.created",
        "customer.subscription.created",
        "invoice.paid",
        "invoice.payment_failed",
    ]
)
WEBHOOK_SECRET = endpoint.secret

print(f"Server running. Public URL: {WEBHOOK_URL}")
print(f"Webhook endpoint: {endpoint.id}")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:4242
INFO:werkzeug:Press CTRL+C to quit


ERROR:pyngrok.process.ngrok:t=2026-04-15T15:20:28+0000 lvl=eror msg="failed to reconnect session" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-04-15T15:20:28+0000 lvl=eror msg="session closing" obj=tunnels.session err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n"
ERROR:pyngrok.process.ngrok:t=2026-04-15T15:20:28+0000 lvl=eror msg="terminating with error" obj=app err="authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your aut

PyngrokNgrokError: The ngrok process errored on start: authentication failed: Usage of ngrok requires a verified account and authtoken.\n\nSign up for an account: https://dashboard.ngrok.com/signup\nInstall your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken\r\n\r\nERR_NGROK_4018\r\n.

---

## Exercise 1: Trigger a Payment Failure Event

Equivalent to: `stripe trigger payment_intent.payment_failed`

In [ ]:
def count_events(event_type):
    return sum(
        1 for e in received_events
        if (e['type'] if isinstance(e, dict) else e.type) == event_type
    )

before = count_events('payment_intent.payment_failed')

try:
    stripe.PaymentIntent.create(
        amount=1500,
        currency="usd",
        payment_method="pm_card_visa_chargeDeclined",
        confirm=True,
        automatic_payment_methods={"enabled": True, "allow_redirects": "never"}
    )
except stripe.error.CardError:
    pass  # Expected — we're testing the failure path

time.sleep(5)

after = count_events('payment_intent.payment_failed')
print(f"payment_intent.payment_failed events received: {after - before} new (total: {after})")

---

## Exercise 2: Trigger a Subscription Created Event

Equivalent to: `stripe trigger customer.subscription.created`

In [ ]:
# Create a product, price, customer and subscription
product = stripe.Product.create(name="Webhook Test Plan")
price   = stripe.Price.create(
    product=product.id,
    unit_amount=1900,
    currency="usd",
    recurring={"interval": "month"}
)
customer = stripe.Customer.create(
    email="webhook-test@example.com",
    payment_method="pm_card_visa",
    invoice_settings={"default_payment_method": "pm_card_visa"}
)

before_sub = count_events('customer.subscription.created')
before_inv = count_events('invoice.paid')

subscription = stripe.Subscription.create(
    customer=customer.id,
    items=[{"price": price.id}]
)

SUB_ID = subscription.id
print(f"Subscription: {SUB_ID} | status: {subscription.status}")
print("Waiting for webhook deliveries...")
time.sleep(8)

print(f"\ncustomer.subscription.created: {count_events('customer.subscription.created') - before_sub} new")
print(f"invoice.paid:                  {count_events('invoice.paid') - before_inv} new")

---

## Exercise 3: Trigger a Refund Event

Equivalent to: `stripe trigger charge.refunded`

In [ ]:
pi = stripe.PaymentIntent.create(
    amount=5000,
    currency="usd",
    payment_method="pm_card_visa",
    confirm=True,
    automatic_payment_methods={"enabled": True, "allow_redirects": "never"}
)

pi_expanded = stripe.PaymentIntent.retrieve(pi.id, expand=["latest_charge"])
refund = stripe.Refund.create(charge=pi_expanded.latest_charge.id)

before = count_events('charge.refunded')
time.sleep(5)

print(f"Refund {refund.id}: {refund.status}")
print(f"charge.refunded events received: {count_events('charge.refunded') - before} new")

---

## Exercise 4: Inspect All Received Events

In [ ]:
from collections import Counter

types = [(e['type'] if isinstance(e, dict) else e.type) for e in received_events]
counts = Counter(types)

print(f"Total events received: {len(received_events)}")
print()
print("By type:")
for event_type, count in sorted(counts.items()):
    print(f"  {event_type:<45} x{count}")

---

## Exercise 5: Inspect Events via the Stripe API

The Stripe API lets you retrieve the event log directly — useful for debugging missed deliveries.

In [ ]:
from datetime import datetime

# List the 10 most recent events across all types
events = stripe.Event.list(limit=10)

print(f"{'Event Type':<45} {'ID':<30} {'Created'}")
print("-" * 95)
for e in events.data:
    created = datetime.fromtimestamp(e.created).strftime('%H:%M:%S')
    print(f"{e.type:<45} {e.id:<30} {created}")

In [ ]:
# Filter to a specific event type and inspect the payload
failed_events = stripe.Event.list(type='payment_intent.payment_failed', limit=1)

if failed_events.data:
    e = failed_events.data[0]
    pi_obj = e.data.object
    print(f"Event:               {e.id}")
    print(f"Type:                {e.type}")
    print(f"PaymentIntent:       {pi_obj.id}")
    print(f"Amount:              ${pi_obj.amount / 100:.2f}")
    if pi_obj.last_payment_error:
        err = pi_obj.last_payment_error
        print(f"\nlast_payment_error:")
        print(f"  code:         {err.code}")
        print(f"  decline_code: {err.decline_code}")
        print(f"  message:      {err.message}")

---

## Using the Dashboard: Inspect & Resend Events

The Dashboard provides a visual event log you can filter, inspect, and replay.

### View Events
1. Go to [Developers → Events](https://dashboard.stripe.com/test/events)
2. Filter by event type — try `payment_intent.payment_failed`
3. Click an event to view the full JSON payload
4. Look at the **`last_payment_error`** field — it contains the decline code

### View Webhook Deliveries
1. Go to [Developers → Webhooks](https://dashboard.stripe.com/test/webhooks)
2. Click your endpoint
3. Under **Recent deliveries**, see each event delivery attempt and its HTTP response code

### Resend an Event
If your handler failed (returned non-200), you can retry:
1. Click any delivery attempt
2. Click **Resend**

Or via the API:

In [ ]:
# Resend the most recent payment_intent.payment_failed event to your endpoint
failed_events = stripe.Event.list(type='payment_intent.payment_failed', limit=1)

if failed_events.data:
    event_to_resend = failed_events.data[0]
    # Use the WebhookEndpoint resend API
    # (replace endpoint.id with your endpoint ID if running fresh)
    stripe.WebhookEndpoint.resend(
        endpoint.id,
        event_to_resend.id
    )
    print(f"Resent event {event_to_resend.id} to {endpoint.id}")
    time.sleep(3)
    print(f"payment_intent.payment_failed total: {count_events('payment_intent.payment_failed')}")
else:
    print("No payment_intent.payment_failed events found.")

---

## Troubleshooting Guide

| Symptom | Likely cause | Fix |
|---------|-------------|-----|
| Events not arriving | Endpoint URL unreachable | Check ngrok tunnel is running |
| 400 response on delivery | Signature verification failed | Confirm `WEBHOOK_SECRET` matches your endpoint's secret |
| `SignatureVerificationError` | Using raw bytes vs parsed JSON | Pass `request.data` (raw bytes) to `construct_event`, not a parsed dict |
| Duplicate processing | Stripe retried after a timeout | Use `event.id` to deduplicate |
| Missing events | Event type not subscribed | Update `enabled_events` on your endpoint |

---

## Cleanup

In [ ]:
try:
    stripe.Subscription.cancel(SUB_ID)
    print(f"Cancelled subscription: {SUB_ID}")
except Exception as e:
    print(f"Could not cancel subscription: {e}")

try:
    stripe.WebhookEndpoint.delete(endpoint.id)
    print(f"Deleted webhook endpoint: {endpoint.id}")
except Exception as e:
    print(f"Could not delete endpoint: {e}")

try:
    from pyngrok import ngrok
    ngrok.disconnect(public_url)
    print("ngrok tunnel closed.")
except Exception as e:
    print(f"Could not close ngrok: {e}")

---

## Summary

- Every `stripe trigger <event>` CLI command has a Python API equivalent — create the underlying resource and the event fires automatically
- `stripe.Event.list()` gives you the same event log as the Dashboard → Events tab
- The Dashboard lets you inspect full payloads, view delivery attempts, and **resend** failed events
- The `last_payment_error` field on a failed PaymentIntent tells you exactly why it failed
- Use `event.id` for idempotency — Stripe may deliver the same event more than once

## Lab 02 Complete!

Continue to `07_terraform_intro.ipynb` to manage Stripe resources as code with Terraform.